# PantheraLAB 🐆 · Laboratorio de IA real v5: SAM 2

**Nuevo modelo SAM 2.1 Small (Meta):** a partir de las cajas reales de MegaDetector y de una región candidata LR-ASPP, intentamos obtener una máscara que incluya más cuerpo del animal. Comparamos varios prompts y predicciones; no inventamos una caja más amplia.

La foto del perro está acostada y MegaDetector marcó principalmente su cabeza. SAM 2 usa también una caja que une cabeza y región candidata de perro, además de puntos de referencia calculados. Por eso tiene una oportunidad de proponer el cuerpo completo, **pero no está garantizado**.

**Secuencia:** MegaDetector V6 Extra → LR-ASPP (referencia semántica) → SAM 2 (máscaras por prompts) → SpeciesNet (taxonomía preliminar) → comparativa e informe.

**Para ejecutar:** en Colab, activá **Entorno de ejecución → Cambiar tipo de entorno → GPU T4** si Google la ofrece. Después **Ejecutar todas** y subí una fotografía autorizada sin personas. SAM 2 se omitirá si no hay GPU; MegaDetector y los modelos anteriores pueden seguir ejecutándose. La instalación y descarga de pesos pueden tardar varios minutos; la GPU gratuita no está garantizada.

**Alcance:** fotografía temporal procesada en Colab, sin integración con Render ni PantheraLIVE y sin identificación científica validada. SAM 2 no identifica especies ni garantiza la silueta completa. No subas imágenes de personas.

Modelos oficiales: https://github.com/facebookresearch/sam2 · https://huggingface.co/facebook/sam2.1-hiera-small · https://github.com/microsoft/MegaDetector · https://github.com/google/cameratrapai

## 1. Instalar modelos en Google Colab

Se instalan MegaDetector, SpeciesNet y **SAM 2 oficial** en esta sesión temporal. Para evitar errores habituales con la extensión CUDA al instalar SAM 2, se desactiva esa extensión opcional; la inferencia en GPU sigue estando disponible. PyTorch y Torchvision deben tener versiones compatibles. Verificá que Colab haya conectado la GPU si querés ejecutar SAM 2.

In [ ]:
%pip -q install PytorchWildlife speciesnet Pillow huggingface_hub
%env SAM2_BUILD_CUDA=0
%pip -q install git+https://github.com/facebookresearch/sam2.git

## 2. Elegir la foto del perro u otro animal
Se ejecuta una única prueba sobre tu fotografía. Recomendación: JPG, PNG o WEBP de hasta 12 MB.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps
from pathlib import Path
from io import BytesIO
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se seleccionó una fotografía.")
name, payload = next(iter(uploaded.items()))
if len(payload)>12*1024*1024:
    raise ValueError("Fotografía mayor que 12 MB.")
photo = ImageOps.exif_transpose(Image.open(BytesIO(payload))).convert("RGB")
if photo.width*photo.height>25_000_000:
    raise ValueError("Fotografía mayor que 25 megapíxeles.")
folder = Path("/content/pantheralab_prueba_real")
folder.mkdir(parents=True,exist_ok=True)
file_path = folder/"fotograma.jpg"
photo.save(file_path,quality=92)
preview = photo.copy()
preview.thumbnail((820,820))
display(preview)
print("Fotografía preparada:",photo.width,"x",photo.height)

## 3. MegaDetector V6 Extra · Detección original

Se analiza **una sola vez la fotografía sin rotarla**, para evitar los falsos positivos de personas y vehículos observados en la prueba anterior. MegaDetector produce cajas de detección, pero no necesita encontrar el cuerpo completo para generar un candidato. Conservamos su puntuación original y no la confundimos con precisión científica.

In [ ]:
import torch
from PIL import ImageDraw
from PytorchWildlife.models import detection as pw_detection

device="cuda" if torch.cuda.is_available() else "cpu"
variant="MDV6-yolov10-e"
threshold=0.20
W,H=photo.size
print("MegaDetector:",variant,"| Procesador:",device)
model=pw_detection.MegaDetectorV6(device=device,pretrained=True,version=variant)
prediction=model.single_image_detection(str(file_path),det_conf_thres=threshold)
objects=prediction.get("detections")
names={0:"animal",1:"person",2:"vehicle"}
detections=[]
if objects is not None:
    for box,score,category in zip(objects.xyxy,objects.confidence,objects.class_id):
        xyxy=[int(round(float(v))) for v in box]
        x1,y1,x2,y2=xyxy
        if x2<=x1 or y2<=y1:
            continue
        detections.append({
            "class_name":names.get(int(category),"unknown"),
            "confidence":round(float(score),4),
            "bbox_xyxy":xyxy,
            "orientation_degrees":0,
            "box_fraction_of_image":round((x2-x1)*(y2-y1)/(W*H),5)
        })
detections.sort(key=lambda item:item["confidence"],reverse=True)
animals=[d for d in detections if d["class_name"]=="animal"]
people=[d for d in detections if d["class_name"]=="person"]
primary_detection=animals[0] if animals else None
person_review_candidates=[d for d in people if d["confidence"]>=0.40]
weak_person_candidates=[d for d in people if d["confidence"]<0.40]

annotated=photo.copy()
draw=ImageDraw.Draw(annotated)
for item in detections:
    box=item["bbox_xyxy"]
    color="#00dc9c" if item is primary_detection else "#ffc36b"
    draw.rectangle(box,outline=color,width=5 if item is primary_detection else 3)
    draw.text((box[0]+3,max(0,box[1]-18)),
              item["class_name"].upper()+" "+str(round(item["confidence"]*100))+"%",
              fill=color)
annotated.save(folder/"detecciones.jpg",quality=92)
overview=annotated.copy()
overview.thumbnail((920,920))
display(overview)
print("Animales:",len(animals),"| Posibles personas:",len(people),
      "| Posibles vehículos:",sum(d["class_name"]=="vehicle" for d in detections))
if primary_detection:
    print("Caja principal:",round(primary_detection["confidence"]*100,2),
          "% de confianza; cubre",round(primary_detection["box_fraction_of_image"]*100,2),
          "% de la foto. Esto NO mide cuánto del cuerpo está cubierto.")
else:
    print("⚠ No se encontró un candidato 'animal'; la segmentación posterior será independiente.")
if weak_person_candidates:
    print("⚠ Detecciones débiles de personas sin validar; revisá la foto antes de usarla.")
if person_review_candidates:
    print("⚠ Posible presencia humana. No usar esta foto para clasificación en este ensayo.")

## 4. Segmentación semántica de referencia · LR-ASPP

Este modelo identifica candidatos de algunas categorías comunes (como perro). Sirve de **pista visual** para SAM 2 y como comparación independiente, pero no detecta jaguares como especie. Conservamos su máscara interna y su caja original; no la presentamos como una segmentación definitiva.

In [ ]:
import numpy as np
import torch.nn.functional as F
from torchvision.models.segmentation import (
    lraspp_mobilenet_v3_large, LRASPP_MobileNet_V3_Large_Weights
)

segmentation_model="torchvision/lraspp_mobilenet_v3_large"
segmentation_status="not_run"
segmentation_candidates=[]
segmentation_selected=None
segmented_crop=None
segment_bbox=None
segmentation_overlay=None
semantic_mask=None

if person_review_candidates:
    segmentation_status="person_review_required"
    print("Segmentación omitida: posible presencia de personas; revisar la foto.")
else:
    try:
        weights=LRASPP_MobileNet_V3_Large_Weights.DEFAULT
        categories=weights.meta["categories"]
        print("Cargando segmentador ligero. Categorías probadas: perro, gato y algunos animales domésticos.")
        segmenter=lraspp_mobilenet_v3_large(weights=weights).to(device).eval()
        tensor=weights.transforms()(photo).unsqueeze(0).to(device)
        with torch.inference_mode():
            raw=segmenter(tensor)["out"]
            probabilities=raw.softmax(dim=1)
            winning=probabilities.argmax(dim=1,keepdim=True)
        class_names=("dog","cat","bird","cow","horse","sheep")
        candidates=[]
        for class_name in class_names:
            if class_name not in categories:
                continue
            class_id=categories.index(class_name)
            # Upscale only the class index map (nearest), avoiding large GPU allocations.
            label_map=F.interpolate(
                winning.float(),size=(H,W),mode="nearest"
            )[0,0].round().to(torch.int64).cpu().numpy()
            mask=(label_map==class_id)
            area=int(mask.sum())
            if area<max(150,int(W*H*0.002)) or area>int(W*H*0.80):
                continue
            ys,xs=np.where(mask)
            x1,y1,x2,y2=int(xs.min()),int(ys.min()),int(xs.max()+1),int(ys.max()+1)
            overlap_fraction=0.0
            if primary_detection:
                ax,ay,bx,by=primary_detection["bbox_xyxy"]
                ax,ay,bx,by=max(ax,0),max(ay,0),min(bx,W),min(by,H)
                if bx>ax and by>ay:
                    overlap_fraction=float(mask[ay:by,ax:bx].mean())
            small_winner=(winning[0,0]==class_id)
            class_probability=(float(probabilities[0,class_id][small_winner].mean().item())
                               if bool(small_winner.any()) else 0.0)
            candidates.append({
                "class_name":class_name,
                "estimated_pixel_score":round(class_probability,4),
                "pixel_fraction_of_image":round(area/(W*H),5),
                "bbox_xyxy":[x1,y1,x2,y2],
                "overlap_with_megadetector":round(overlap_fraction,4),
                "_mask":mask
            })
        segmentation_candidates=[{
            k:v for k,v in candidate.items() if k!="_mask"
        } for candidate in candidates]
        plausible=[c for c in candidates
                   if primary_detection and c["overlap_with_megadetector"]>=0.10]
        plausible.sort(key=lambda c:(c["overlap_with_megadetector"],
                                     c["estimated_pixel_score"]),reverse=True)
        if plausible:
            chosen=plausible[0]
            segmentation_selected={k:v for k,v in chosen.items() if k!="_mask"}
            semantic_mask=chosen["_mask"]
            segment_bbox=chosen["bbox_xyxy"]
            mask_image=Image.fromarray((chosen["_mask"].astype(np.uint8)*255),mode="L")
            # Green-tinted mask is a genuine output of the segmenter, not an expanded box.
            tint=Image.new("RGB",photo.size,(18,209,146))
            blended=Image.blend(photo,tint,alpha=0.48)
            segmentation_overlay=Image.composite(blended,photo,mask_image)
            draw_seg=ImageDraw.Draw(segmentation_overlay)
            draw_seg.rectangle(segment_bbox,outline="#fff6be",width=5)
            draw_seg.text((segment_bbox[0]+5,max(0,segment_bbox[1]-19)),
                          chosen["class_name"].upper()+" · SEGMENTACIÓN EXPERIMENTAL",
                          fill="#fff6be")
            segmentation_overlay.save(folder/"segmentacion_animal.jpg",quality=92)
            segmented_crop=photo.crop(tuple(segment_bbox))
            segmentation_status="candidate"
            print("Silueta candidata:",chosen["class_name"],
                  "| puntuación media de píxeles:",round(chosen["estimated_pixel_score"]*100,1),"%",
                  "| superficie:",round(chosen["pixel_fraction_of_image"]*100,2),"% de la foto",
                  "| coincidencia con caja MD:",round(chosen["overlap_with_megadetector"]*100,1),"%")
            print("⚠ La máscara puede incluir fondo o partes incorrectas. No es una especie confirmada.")
        else:
            segmentation_status="no_associated_mask"
            print("No apareció una máscara plausible asociada con MegaDetector.")
            print("No se inventó ni amplió una caja: conservamos el resultado original.")
    except Exception as exc:
        segmentation_status="error"
        print("El segmentador no pudo ejecutarse:",type(exc).__name__,str(exc)[:240])
        print("MegaDetector y SpeciesNet aún pueden ejecutarse con sus recortes propios.")

## 5. SAM 2.1 Small · Refinar el contorno del animal (NUEVO)

El predictor oficial SAM 2 recibe la fotografía completa y varios **prompts generados a partir de resultados reales**:

1. Recuadro de MegaDetector (cabeza; referencia).
2. Caja unificada de MegaDetector y LR-ASPP (cuerpo + cabeza, **solo como prompt**, no como detección).
3. Caja unificada más un punto dentro de la silueta semántica y un punto dentro del recuadro de MegaDetector.

SAM 2 devuelve varias máscaras y una **puntuación estimada de calidad de máscara**, que NO representa probabilidad de que sea un perro ni exactitud científica. Se selecciona una candidata plausible por tamaño, coincidencia con MegaDetector y concordancia con LR-ASPP. La máscara puede seguir incluyendo sofá o dejar cuerpo fuera.

Si no hay GPU, no hay detección previa o la instalación falla, registra un estado explícito sin fabricar resultados. Para el perro, la comparación será útil, pero **todavía debemos validar cualquier mejora con imágenes etiquetadas**.

In [ ]:
import numpy as np
from PIL import ImageDraw
from contextlib import nullcontext

sam2_model_id="facebook/sam2.1-hiera-small"
sam2_status="not_run"
sam2_error=None
sam2_candidates=[]
sam2_selected=None
sam2_overlay=None
sam2_crop=None
sam2_mask=None
sam2_mask_bbox=None
sam2_prompts=[]

def union_boxes(first,second):
    return [
        max(0,min(int(first[0]),int(second[0]))),
        max(0,min(int(first[1]),int(second[1]))),
        min(W,max(int(first[2]),int(second[2]))),
        min(H,max(int(first[3]),int(second[3])))
    ]

def mask_bbox_coordinates(mask):
    ys,xs=np.nonzero(mask)
    if len(xs)==0:
        return None
    return [int(xs.min()),int(ys.min()),int(xs.max()+1),int(ys.max()+1)]

if person_review_candidates:
    sam2_status="person_review_required"
    print("SAM 2 omitido: detección humana que necesita revisión.")
elif primary_detection is None:
    sam2_status="no_animal_prompt"
    print("SAM 2 omitido: MegaDetector no encontró animal para guiarlo.")
elif not torch.cuda.is_available():
    sam2_status="gpu_required"
    print("SAM 2 omitido: activá GPU T4 en Entorno de ejecución → Cambiar tipo de entorno.")
else:
    try:
        from sam2.sam2_image_predictor import SAM2ImagePredictor

        main_box=list(primary_detection["bbox_xyxy"])
        prompt_definitions=[{
            "name":"solo_megadetector",
            "box":main_box,
            "points":None,
            "point_labels":None
        }]
        if segmentation_selected is not None:
            body_box=list(segmentation_selected["bbox_xyxy"])
            combined=union_boxes(main_box,body_box)
            prompt_definitions.append({
                "name":"union_megadetector_y_lraspp",
                "box":combined,
                "points":None,
                "point_labels":None
            })
            # Point belonging to LR-ASPP's dog pixels, preferably away
            # from the face detection. This is only an automatic prompt.
            positive_body=None
            if semantic_mask is not None:
                ys,xs=np.nonzero(semantic_mask)
                if len(xs)>0:
                    face_y0=int(main_box[1])
                    far=(ys<face_y0)
                    pool=np.flatnonzero(far) if np.any(far) else np.arange(len(xs))
                    median_y=float(np.median(ys[pool]))
                    band=pool[np.abs(ys[pool]-median_y)<max(14,H*0.06)]
                    if len(band)>0:
                        pool=band
                    center_x=float(np.median(xs[pool]))
                    j=int(pool[np.argmin(np.abs(xs[pool]-center_x))])
                    positive_body=[int(xs[j]),int(ys[j])]
            if positive_body is not None:
                # A detected head is NOT necessarily centered on dog
                # pixels; only add head point when inside predicted mask.
                face_center=[
                    int((main_box[0]+main_box[2])/2),
                    int((main_box[1]+main_box[3])/2)
                ]
                prompt_definitions.append({
                    "name":"union_con_punto_en_cuerpo",
                    "box":combined,
                    "points":[positive_body],
                    "point_labels":[1]
                })
                print("Punto de cuerpo sugerido por LR-ASPP:",positive_body)

        sam2_prompts=[{
            "name":prompt["name"],
            "box_prompt_xyxy":prompt["box"],
            "positive_points_xy":prompt["points"] or []
        } for prompt in prompt_definitions]
        print("Descargando/cargando SAM 2.1 Small en GPU. Puede tardar…")
        predictor=SAM2ImagePredictor.from_pretrained(sam2_model_id,device="cuda")
        predictor.set_image(np.asarray(photo.convert("RGB")))
        mask_store=[]
        for prompt in prompt_definitions:
            coords=(np.asarray(prompt["points"],dtype=np.float32)
                    if prompt["points"] else None)
            labels=(np.asarray(prompt["point_labels"],dtype=np.int32)
                    if prompt["point_labels"] else None)
            masks,estimated_ious,_=predictor.predict(
                box=np.asarray(prompt["box"],dtype=np.float32),
                point_coords=coords,
                point_labels=labels,
                multimask_output=True
            )
            for i,(mask,estimated_iou) in enumerate(zip(masks,estimated_ious)):
                binary=np.asarray(mask,dtype=bool)
                fraction=float(binary.mean())
                bounded=mask_bbox_coordinates(binary)
                if bounded is None:
                    continue
                x1,y1,x2,y2=main_box
                md_support=float(binary[y1:y2,x1:x2].mean()) if x2>x1 and y2>y1 else 0
                semantic_support=None
                if semantic_mask is not None:
                    positive=int(np.count_nonzero(semantic_mask))
                    semantic_support=(float(np.count_nonzero(binary & semantic_mask)/positive)
                                      if positive else None)
                quality=float(estimated_iou)
                plausible=(0.003<=fraction<=0.55 and quality>=0.45 and md_support>=0.08)
                if semantic_support is not None and semantic_support<0.10:
                    plausible=False
                # This ranking is a visual heuristic, not model confidence.
                ranking=(
                    0.70*quality+0.15*md_support+
                    0.10*(semantic_support if semantic_support is not None else 0)+
                    0.05*min(fraction/0.25,1.0)
                )
                record={
                    "prompt":prompt["name"],
                    "mask_index":i,
                    "sam2_predicted_iou":round(quality,4),
                    "pixel_fraction_of_image":round(fraction,5),
                    "bbox_xyxy":bounded,
                    "fraction_of_md_box_covered":round(md_support,4),
                    "fraction_of_lraspp_pixels_covered":(
                        round(semantic_support,4) if semantic_support is not None else None
                    ),
                    "selection_score_heuristic":round(ranking,4),
                    "plausible":bool(plausible)
                }
                sam2_candidates.append(record)
                mask_store.append(binary)
                print(prompt["name"],"máscara",i,
                      "| IoU estimada SAM 2:",round(quality,3),
                      "| área:",round(fraction*100,2),"%",
                      "| pasa filtro:",plausible)
        valid=[(k,rec) for k,rec in enumerate(sam2_candidates) if rec["plausible"]]
        if not valid:
            sam2_status="no_plausible_mask"
            print("⚠ SAM 2 generó propuestas, pero ninguna pasó los filtros. No se simula un resultado.")
        else:
            best_index,best=max(valid,key=lambda item:item[1]["selection_score_heuristic"])
            sam2_selected=dict(best)
            sam2_mask=mask_store[best_index]
            sam2_mask_bbox=list(best["bbox_xyxy"])
            sam2_crop=photo.crop(tuple(sam2_mask_bbox))
            tint=Image.new("RGB",photo.size,(34,196,246))
            alpha_mask=Image.fromarray(sam2_mask.astype(np.uint8)*255,mode="L")
            # Blue pixels are the SAM 2 model mask, not the union prompt.
            sam2_overlay=Image.composite(
                Image.blend(photo,tint,alpha=0.55),photo,alpha_mask
            )
            outline=ImageDraw.Draw(sam2_overlay)
            outline.rectangle(sam2_mask_bbox,outline="#fff4a8",width=5)
            outline.text(
                (sam2_mask_bbox[0]+5,max(0,sam2_mask_bbox[1]-20)),
                "SAM 2 · CONTORNO CANDIDATO",fill="#fff4a8"
            )
            sam2_overlay.save(folder/"segmentacion_sam2.jpg",quality=93)
            mask_file=folder/"mascara_sam2.png"
            alpha_mask.save(mask_file)
            sam2_status="candidate"
            print("SAM 2: máscara candidata (NO validada).",
                  "Caja:",sam2_mask_bbox,
                  "IoU estimada:",best["sam2_predicted_iou"])
            display(sam2_overlay.resize((min(W,800),round(H*min(W,800)/W))))
        del predictor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as exc:
        sam2_status="error"
        sam2_error=type(exc).__name__+": "+str(exc)[:350]
        print("SAM 2 no pudo ejecutarse:",sam2_error)
        print("MegaDetector, LR-ASPP y SpeciesNet conservan sus resultados.")

print("SAM 2:",sam2_status)

## 6. SpeciesNet · ¿Qué sugiere sobre las regiones detectadas?

SpeciesNet compara recortes reales: MegaDetector, contexto, LR-ASPP y **SAM 2 si obtuvo una máscara plausible**. Ninguna sugerencia constituye identificación científica. Un recorte grande puede incluir fondo o partes del sofá; conservamos la procedencia de cada región.

También se muestra una comparación final de la imagen completa, la caja original, LR-ASPP y SAM 2. La máscara azul es la que predice SAM 2, **no un rectángulo agrandado artificialmente**.

In [ ]:
from speciesnet.classifier import SpeciesNetClassifier
import kagglehub
import json
from pathlib import Path
from shutil import copy2

species_suggestions=[]
species_status="not_run"
species_note=""
species_views=[]
exact_crop=None
context_crop=None

def prepare_classifier_model():
    official_id="google/speciesnet/pyTorch/v4.0.3a/1"
    original=Path(kagglehub.model_download(official_id)).resolve()
    metadata=json.loads((original/"info.json").read_text(encoding="utf-8"))
    target=folder/"speciesnet_classifier_only"
    target.mkdir(parents=True,exist_ok=True)
    for key in ("classifier","classifier_labels"):
        if key not in metadata:
            raise RuntimeError("El modelo oficial no contiene "+key)
        relative=Path(metadata[key])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Ruta de pesos inesperada: "+str(relative))
        source_file=original/relative
        if not source_file.is_file():
            raise FileNotFoundError(str(source_file))
        target_file=target/relative
        target_file.parent.mkdir(parents=True,exist_ok=True)
        if not target_file.exists():
            try:
                target_file.symlink_to(source_file)
            except OSError:
                copy2(source_file,target_file)
    metadata["detector"]="unused_in_classifier_only_test.pt"
    (target/"info.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
    return target

if person_review_candidates:
    species_status="person_review_required"
    species_note="Hay una señal de persona suficientemente fuerte para requerir revisión manual. No se clasifica esta imagen."
    print("⚠ Revisá si la imagen contiene personas. Evitá subir imágenes de personas a Colab.")
elif weak_person_candidates:
    print("Aviso: el detector produjo señales débiles de persona en vistas giradas.",
          "No se interpretan automáticamente como presencia humana ni prueba de ausencia de personas.")
if species_status=="not_run" and primary_detection is None:
    species_status="no_animal"
    species_note="MegaDetector no encontró animales."
elif species_status=="not_run" and primary_detection is not None and primary_detection["box_fraction_of_image"]<0.01:
    species_status="insufficient_bbox"
    species_note="Caja principal demasiado pequeña para esta prueba de clasificación."
elif species_status=="not_run":
    x1,y1,x2,y2=primary_detection["bbox_xyxy"]
    exact_crop=photo.crop((x1,y1,x2,y2))
    w,h=x2-x1,y2-y1
    margin=0.65
    padded_bbox=(max(0,int(x1-w*margin)),max(0,int(y1-h*margin)),
                 min(W,int(x2+w*margin)),min(H,int(y2+h*margin)))
    context_crop=photo.crop(padded_bbox)

    try:
        print("Cargando clasificador SpeciesNet…")
        local_model_dir=prepare_classifier_model()
        classifier=SpeciesNetClassifier(model_name=str(local_model_dir),device=device)
        input_views=[("recuadro_del_detector",exact_crop),
                     ("recorte_con_contexto",context_crop)]
        if segmented_crop is not None:
            input_views.append(("caja_de_segmentacion",segmented_crop))
        if sam2_crop is not None:
            input_views.append(("silueta_sam2",sam2_crop))
        for crop_name,crop_image in input_views:
            result=classifier.predict(crop_name+".jpg",classifier.preprocess(crop_image))
            classifications=result.get("classifications",{})
            suggestions=[{
                "label":str(label).rsplit(";",1)[-1].strip(),
                "raw_label":str(label),
                "score":round(float(score),4)
            } for label,score in zip(
                classifications.get("classes",[]),
                classifications.get("scores",[])
            )][:5]
            top=suggestions[0]["label"].lower() if suggestions else ""
            classification="not_identified" if top=="blank" else (
                "suggested" if suggestions else "no_prediction")
            species_views.append({
                "crop":crop_name,
                "bbox_xyxy":(
                    list(primary_detection["bbox_xyxy"]) if crop_name=="recuadro_del_detector"
                    else list(segment_bbox) if crop_name=="caja_de_segmentacion"
                    else list(sam2_mask_bbox) if crop_name=="silueta_sam2"
                    else list(padded_bbox)
                ),
                "region_provenance":(
                    "megadetector_box" if crop_name=="recuadro_del_detector"
                    else "lraspp_semantic_mask_bbox" if crop_name=="caja_de_segmentacion"
                    else "sam2_prompted_mask_bbox" if crop_name=="silueta_sam2"
                    else "expanded_context_not_model_detection"
                ),
                "model_generated_bbox":crop_name in ("recuadro_del_detector","caja_de_segmentacion","silueta_sam2"),
                "status":classification,
                "suggestions":suggestions
            })
            print("\n"+crop_name.replace("_"," ").upper())
            for suggestion in suggestions[:3]:
                print("  ",suggestion["label"],round(suggestion["score"]*100,2),"%",
                      "— sugerencia sin validar")
        valid=[view for view in species_views if view["status"]=="suggested"]
        if valid:
            species_status="suggested"
            # El recorte exacto es el predeterminado; el ampliado solo añade contexto.
            preferred=next((view for view in valid if view["crop"]=="recuadro_del_detector"),valid[0])
            species_suggestions=preferred["suggestions"]
            species_note="Sugerencias reales; comparar las dos vistas y confirmar manualmente."
        else:
            species_status="not_identified"
            species_note="Ningún recorte produjo una especie reconocible."
    except Exception as exc:
        species_status="error"
        species_note=type(exc).__name__+": "+str(exc)[:230]
        print("No se pudo finalizar SpeciesNet:",species_note)

print("\nESTADO:",species_status,"·",species_note)

# Presentación explícita: foto original COMPLETA + detección real + recortes.
from PIL import ImageFont
panels=[("FOTO COMPLETA · original",photo),
        ("DETECCIONES REALES · cajas del modelo",annotated)]
if exact_crop is not None:
    panels.append(("RECORTE EXACTO · de MegaDetector",exact_crop))
if context_crop is not None:
    panels.append(("CONTEXTO EXTRA · NO es detección",context_crop))
if segmentation_overlay is not None:
    panels.append(("SEGMENTACIÓN · silueta candidata",segmentation_overlay))
if segmented_crop is not None:
    panels.append(("REGIÓN SEGMENTADA · NO es MegaDetector",segmented_crop))
if sam2_overlay is not None:
    panels.append(("SAM 2 · MÁSCARA AZUL · candidato",sam2_overlay))
if sam2_crop is not None:
    panels.append(("SAM 2 · RECORTE PARA SPECIESNET",sam2_crop))
panel_width=640
panel_pics=[]
for title,img in panels:
    preview=img.copy()
    preview.thumbnail((panel_width,660))
    background=Image.new("RGB",(panel_width,preview.height+46),"#101e2b")
    overlay=ImageDraw.Draw(background)
    overlay.text((12,14),title,fill="white")
    background.paste(preview,((panel_width-preview.width)//2,40))
    panel_pics.append(background)
comparison=Image.new("RGB",(panel_width,sum(i.height for i in panel_pics)+12*(len(panel_pics)-1)),"#102636")
offset=0
for image in panel_pics:
    comparison.paste(image,(0,offset))
    offset+=image.height+12
comparison.save(folder/"comparacion_visual_pantheralab.jpg",quality=90)
display(comparison)
print("Comparativa guardada: original, MegaDetector, LR-ASPP, SAM 2 y SpeciesNet.")
print("La segmentación es experimental; no reemplaza ni valida la caja de MegaDetector.")

## 7. Evidencia científica y archivos descargables

Se genera un JSON con resultados de MegaDetector, LR-ASPP, SAM 2 y SpeciesNet, las cajas con su **origen específico** y la **IoU estimada de SAM 2** (no precisión real). Se incluyen además la foto completa anotada, la comparación visual y, cuando exista, la máscara binaria de SAM 2 en PNG. Un estado 'gpu_required', 'error' o 'no_plausible_mask' se registra sin fabricar resultados.

Este cuaderno **no publica eventos en PantheraLIVE** ni garantiza que la segmentación de SAM 2 sea correcta. Se necesita revisión científica y un conjunto de imágenes etiquetadas para medir aciertos.

In [ ]:
from datetime import datetime, timezone
import json
summary={
    "platform":"PantheraLAB",
    "test":"Colab MegaDetector V6 Extra + LRASPP + SAM 2.1 Small + SpeciesNet",
    "timestamp":datetime.now(timezone.utc).isoformat(),
    "simulation":False,
    "scientifically_verified":False,
    "source":"authorized_image_uploaded_to_colab",
    "image_dimensions":{"width":W,"height":H},
    "megadetector":{
        "model":variant,"threshold":threshold,"orientations_tested":[0],
        "detections":detections,
        "weak_person_predictions":len(weak_person_candidates),
        "person_review_candidates":len(person_review_candidates),
        "person_gate_policy":"experimental: original 0deg >=0.35 or any view >=0.60; not a privacy guarantee",
        "primary_detection":primary_detection,
        "primary_box_selection":"highest confidence animal, original orientation only"
    },
    "segmentation":{
        "model":segmentation_model,
        "status":segmentation_status,
        "candidates":segmentation_candidates,
        "selected":segmentation_selected,
        "trained_for_jaguars":False,
        "mask_scientifically_validated":False
    },
    "sam2":{
        "model":sam2_model_id,
        "status":sam2_status,
        "error":sam2_error,
        "prompts":sam2_prompts,
        "candidates":sam2_candidates,
        "selected":sam2_selected,
        "mask_is_scientifically_validated":False,
        "mask_pixels_saved_only_if_candidate":sam2_status=="candidate"
    },
    "speciesnet":{
        "status":species_status,"note":species_note,
        "suggestions":species_suggestions,"views":species_views,
        "context_bbox_is_model_detection":False,
        "segmentation_region_is_megadetector_bbox":False
    },
    "saved_in_pantheralive":False
}
report_file=folder/"resultado_pantheralab.json"
report_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print("RESULTADO:")
print(json.dumps(summary,ensure_ascii=False,indent=2)[:12000])
print("Estas probabilidades son puntuaciones de modelos, no identificaciones validadas.")

In [ ]:
from google.colab import files
files.download(str(report_file))
files.download(str(folder/"detecciones.jpg"))
files.download(str(folder/"comparacion_visual_pantheralab.jpg"))
if segmentation_overlay is not None:
    files.download(str(folder/"segmentacion_animal.jpg"))
if sam2_overlay is not None:
    files.download(str(folder/"segmentacion_sam2.jpg"))
    files.download(str(folder/"mascara_sam2.png"))

**Lectura correcta:** MegaDetector detecta un posible animal, LR-ASPP propone categorías de animales comunes, SAM 2 refina contornos a partir de cajas o puntos y SpeciesNet propone taxonomía. **SAM 2 no identifica especies, individuos ni distingue jaguar por sí solo.**

**Limitaciones:** SAM 2 Small suele necesitar GPU Colab; no se ha ejecutado este cuaderno en tu sesión ni con tu perro. El recuadro inicial de cabeza puede arrastrar sesgos a la máscara aunque añadamos una caja compuesta y un punto auxiliar. La calidad debe comprobarse con la imagen original y anotación humana, no solo con una puntuación interna. Esta prueba no instala ningún modelo en Render ni modifica PantheraLIVE en producción.

Documentación oficial: https://github.com/facebookresearch/sam2 y https://huggingface.co/facebook/sam2.1-hiera-small